In [1]:
import sqlite3
import pandas as pd

con = sqlite3.connect("../olist.db")

def consulta(sql):
    return pd.read_sql(sql, con)

In [2]:
consulta("""
SELECT COUNT(*) AS vendedores_cadastrados,
       (SELECT COUNT(DISTINCT seller_id) FROM order_items) AS vendedores_com_venda
FROM sellers
""")

,vendedores_cadastrados,vendedores_com_venda
0,3095,3095


In [3]:
consulta("""
SELECT seller_state AS estado, COUNT(*) AS vendedores
FROM sellers
GROUP BY seller_state
ORDER BY vendedores DESC
""")

,estado,vendedores
0,SP,1849
1,PR,349
2,MG,244
3,SC,190
4,RJ,171
5,RS,129
6,GO,40
7,DF,30
8,ES,23
9,BA,19


Todos os 3.095 vendedores cadastrados aparecem em pelo menos um item vendido, então não precisei filtrar vendedores sem venda nas próximas consultas.

Por estado, os vendedores se concentram bastante: SP tem 1.849 (cerca de 60% do total) e os seis primeiros estados (SP, PR, MG, SC, RJ e RS) juntos reúnem cerca de 95%. É mais concentrado do que os clientes, onde SP tinha 41,9%. Os estados do Norte e do Nordeste têm poucos vendedores, quase sempre menos de dez cada.

In [5]:
consulta("""
SELECT oi.seller_id,
       s.seller_state AS estado,
       COUNT(DISTINCT oi.order_id) AS pedidos,
       ROUND(SUM(oi.price), 2) AS valor_vendido
FROM order_items oi
JOIN orders o ON o.order_id = oi.order_id
JOIN sellers s ON s.seller_id = oi.seller_id
WHERE o.order_status NOT IN ('canceled', 'unavailable')
GROUP BY oi.seller_id, s.seller_state
ORDER BY valor_vendido DESC
LIMIT 10
""")

,seller_id,estado,pedidos,valor_vendido
0,4869f7a5dfa277a7dca6462dcf3b52b2,SP,1131,229237.63
1,53243585a1d6dc2643021fd1853d8905,BA,358,222776.05
2,4a3ca9315b744ce9f8e9374361493884,SP,1804,200326.12
3,fa1c13f2614d7b5c4749cbc52fecda94,SP,584,192842.13
4,7c67e1448b00f6e969d365cea6b010ab,SP,982,187923.89
5,7e93a43ef30c4f03f38b393420bc753a,SP,332,172583.88
6,da8622b14eb17ae2831f4ac5b9dab84a,SP,1314,160236.57
7,7a67c85e85bb2ce8582c35f2203ad736,SP,1159,141715.54
8,1025f0e2d44d7041d6cf58b6550e0bfa,SP,915,138968.55
9,955fee9216a65b617aa5c0531780ce60,SP,1286,135159.70


O ranking por valor vendido é dominado por vendedores de SP: nove dos dez primeiros são de lá. A exceção é o segundo colocado, da BA, com R$ 222 mil, quase igual ao primeiro (R$ 229 mil).

Olhando o valor por pedido, os perfis são bem diferentes. O primeiro vendeu em torno de R$ 203 por pedido, o da BA chegou a quase R$ 620 com apenas 358 pedidos, e o terceiro tem o maior volume do top 10 (1.804 pedidos) com cerca de R$ 111 por pedido. Ou seja, dá para chegar ao topo vendendo muito de coisas baratas ou pouco de coisas caras, e o ranking somente por valor esconde essa diferença.

In [6]:
consulta("""
WITH vendas AS (
    SELECT oi.seller_id, SUM(oi.price) AS valor
    FROM order_items oi
    JOIN orders o ON o.order_id = oi.order_id
    WHERE o.order_status NOT IN ('canceled', 'unavailable')
    GROUP BY oi.seller_id
),
acumulado AS (
    SELECT seller_id,
           valor,
           ROW_NUMBER() OVER (ORDER BY valor DESC) AS posicao,
           SUM(valor) OVER (
               ORDER BY valor DESC
               ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
           ) AS valor_acumulado
    FROM vendas
)
SELECT posicao,
       ROUND(valor, 2) AS valor,
       ROUND(100.0 * valor_acumulado / (SELECT SUM(valor) FROM vendas), 1) AS pct_acumulado
FROM acumulado
WHERE posicao IN (1, 10, 50, 100, 300, 500, 1000, 2000)
ORDER BY posicao
""")

,posicao,valor,pct_acumulado
0,1,229237.63,1.7
1,10,135159.70,13.2
2,50,43517.50,33.0
3,100,25080.03,45.3
4,300,9669.80,67.1
5,500,5966.54,78.4
6,1000,2061.37,91.6
7,2000,371.20,98.8


O faturamento é concentrado, mas sem um vendedor dominante: o maior sozinho responde por 1,7% do total e os 10 maiores por 13,2%. A concentração aparece em grupos maiores. Os 100 maiores vendedores (cerca de 3% dos 3.095) reúnem 45,3% do valor vendido, e os 500 maiores (cerca de 16%) chegam a 78,4%.

Na outra ponta, os vendedores depois da posição 1.000 somam só 8,4% do total, e os que estão depois da posição 2.000 dividem 1,2%. Ou seja, boa parte dos vendedores da base vende muito pouco.